---
title: "Chương 2: Phương pháp thu thập dữ liệu"
short_title: "2. Phương pháp thu thập dữ liệu"
---

Chương này mô tả cách crawler hoạt động, công cụ và thư viện được dùng ở từng bước. Toàn bộ mã nguồn nằm trong thư mục `src/` của repository [group11-arxiv-project](https://github.com/mituleefam/group11-arxiv-project); `README.md` của repository hướng dẫn cài đặt và chạy lại.

## 1. Tổng quan pipeline

`main.py` chạy các bước theo thứ tự cho một khoảng ID:

```{mermaid}
flowchart LR
    R["Khoảng ID được giao<br>start_month start_id end_month end_id"] --> D["discovery.py<br>arXiv API id_list"]
    D --> M["metadata.py<br>arXiv API, ID có phiên bản"]
    M --> DL["downloader.py + extractor.py<br>export.arxiv.org/src/&lt;id&gt;v&lt;N&gt;"]
    M --> RF["references.py<br>Semantic Scholar Graph API"]
    DL --> C[("corpus/&lt;yymm-id&gt;/")]
    RF --> C
    C --> V["validate.py<br>corpus_statistics.py"]
```

Bước tải mã nguồn và bước lấy references **chạy song song**, vì chúng gọi hai máy chủ khác nhau (arXiv và Semantic Scholar), mỗi máy chủ có giới hạn request riêng. Lỗi tạm thời không được retry ngay tại chỗ mà được gom lại: hết một lượt (pass), `main.py` chạy thêm lượt retry chỉ cho những bước còn việc. Các kết quả cuối cùng (phiên bản bị rút trả 404, phiên bản chỉ có PDF) không bao giờ bị retry.

## 2. Công cụ và thư viện

| Thành phần | Công cụ | Vai trò |
| :--- | :--- | :--- |
| Discovery, metadata | arXiv API `https://export.arxiv.org/api/query` (Atom XML), `requests`, `xml.etree.ElementTree` | Tra cứu theo `id_list`, đọc hậu tố phiên bản mới nhất, lấy metadata từng phiên bản |
| Mã nguồn | `https://export.arxiv.org/src/<id>v<N>`, `requests`, `tarfile`, `gzip`, `zipfile` | Tải từng phiên bản, nhận diện định dạng, trích xuất `.tex`/`.bib` |
| References | Semantic Scholar Graph API `/paper/batch`, `/paper/{id}/references` | Danh sách reference, `externalIds.ArXiv`, Semantic Scholar ID |
| Ngày nộp của reference | arXiv API, hoặc bản metadata arXiv trên Kaggle (`ref_dates_kaggle.py`) | Điền `ref_submission_date` (ngày nộp `v1`) cho reference có arXiv ID |
| Kiểm tra, đo đạc | `psutil`, `time.perf_counter` | RAM của cây tiến trình, dung lượng thư mục dữ liệu, wall time |
| Báo cáo | `pandas`, `plotly`, Jupyter Book | Bảng và biểu đồ sinh khi build sách |

Nhóm không dùng thư viện `arxiv.py`: crawler gọi thẳng các endpoint bằng `requests`, nhờ đó điều khiển được nhịp request, timeout và retry của mọi bước qua một bộ giới hạn dùng chung (`rate_limiter.py`).

## 3. Entry discovery (`discovery.py`)

1. Sinh danh sách arXiv ID từ 4 cột của bảng đăng ký (`start_month start_id end_month end_id`). Khoảng có thể trải qua nhiều tháng; ID từ năm 2015 có 5 chữ số sau dấu chấm.
2. Gọi arXiv API với `id_list` theo lô **100 ID mỗi lần** và `max_results` bằng đúng kích thước lô (API mặc định chỉ trả 10 kết quả).
3. Kết quả trả về không được giả định theo thứ tự đầu vào: mỗi entry được ghép lại với ID theo mã định danh của chính nó.
4. Mã định danh trả về có hậu tố phiên bản mới nhất (ví dụ `2604.05535v3`), từ đó suy ra danh sách `v1 … v3` cần tải.
5. Mỗi ID được ghi một dòng vào `work/paper_manifest.jsonl` với trạng thái `success`, `not_found` hoặc `api_error`. Một lô lỗi được thử lại ở lượt sau, không bị ghi nhầm thành "không tồn tại".

## 4. Metadata (`metadata.py`)

`metadata.json` của mỗi bài có các trường của bảng `papers`: `title`, `authors`, `submission_date` (ngày nộp `v1`), `revised_dates` (ngày của `v2, v3, …`), `venue` (`journal_ref`, để `null` khi không có). Ngày được chuẩn hóa theo ISO 8601. Metadata của **từng phiên bản** cũng được lưu trong danh sách `versions`. Phiên bản mới nhất lấy lại từ lời gọi discovery, các phiên bản cũ hơn được tra bằng ID có phiên bản (`2604.05535v1`), 100 ID mỗi lần gọi.

## 5. Tải và trích xuất mã nguồn (`downloader.py`, `extractor.py`, `checkpoint.py`)

Mỗi phiên bản được xử lý qua các bước:

1. Tải `https://export.arxiv.org/src/<id>v<N>` (không bao giờ dùng `arxiv.org`) với User-Agent mô tả rõ, có timeout, ghi dần ra file tạm `work/downloads/*.part`.
2. Kiểm tra mã HTTP và đối chiếu kích thước nhận được với `Content-Length`.
3. **Nhận diện định dạng thật** từ các byte đầu của file (magic number), không mặc định là `.tar.gz`:

| Định dạng phát hiện | Xử lý |
| :--- | :--- |
| `tar.gz`, `tar`, `zip` | Trích xuất các thành viên `.tex`/`.bib` |
| `gzip-tex`, `plain-tex` | Cả bài là một file `.tex` |
| `pdf` | Không có mã nguồn: trạng thái `pdf_only`, thư mục rỗng |
| `docx`, `postscript` | Không có mã nguồn TeX: trạng thái `no_tex`, thư mục rỗng |
| trang HTML | Trang lỗi của máy chủ: trạng thái `failed`, thử lại ở lượt sau |

4. Trích xuất vào thư mục tạm `corpus/<yymm-id>/.<vN>.tmp/`, **chỉ ghi file `.tex` và `.bib`**, giữ nguyên đường dẫn tương đối và nội dung từng byte. Đường dẫn tuyệt đối, thành phần `..` và link trong gói nén bị từ chối, để gói nén không thể ghi ra ngoài thư mục phiên bản.
5. Kiểm tra các file đã ghi, đổi tên thư mục tạm thành `v<N>/`, rồi mới ghi trạng thái vào `versions.json`. Nếu chương trình dừng giữa chừng, chỉ còn lại file `.part`/`.tmp`, lần chạy sau xóa và làm lại. Phiên bản đã hoàn tất không bao giờ bị tải hay ghi đè lần nữa.

Kích thước được đo trên **cùng một cơ sở là dung lượng chưa nén**: `original_bytes` là tổng dung lượng mọi file trong gói nguồn, `kept_bytes` là tổng dung lượng các file `.tex`/`.bib` được giữ, nên `1 − kept_bytes / original_bytes` là phần dung lượng bỏ đi. Dung lượng file nén tải về (`downloaded_bytes`) được ghi riêng.

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils import (COLORS, FINAL_SOURCE, REF_OK, group, human_bytes, human_seconds,
                          load, members, meta, metric, mib, pct, provenance, show, table)
provenance()

Số phiên bản theo định dạng mà `extractor.py` phát hiện được trên toàn corpus của nhóm:

In [ ]:
versions = load("versions")
fmt = (versions.assign(source_format=versions["source_format"].fillna("(chưa xác định)"))
       .groupby(["source_format", "status"]).size().reset_index(name="versions")
       .sort_values("versions", ascending=False))
fig = px.bar(fmt, x="source_format", y="versions", color="status", color_discrete_sequence=COLORS,
             title="Số phiên bản theo định dạng nguồn và trạng thái cuối",
             labels={"source_format": "Định dạng phát hiện", "versions": "Số phiên bản", "status": "Trạng thái"})
show(fig)

## 6. References (`references.py`)

Semantic Scholar được gọi với tiền tố `arXiv:<id>`, theo ba bước để dùng ít request nhất:

1. `/paper/batch` với `paperId,referenceCount`, tối đa 500 bài mỗi lần: biết bài nào được lập chỉ mục và có bao nhiêu reference.
2. `/paper/batch` với các trường reference (`title`, `authors`, `year`, `publicationDate`, `venue`, `externalIds`): lấy reference của nhiều bài trong một lần gọi, gom bài sao cho mỗi lần gọi xin tối đa 8 000 reference. Endpoint này chỉ trả tối đa 9 999 reference mỗi lần và **không báo lỗi khi vượt**, nên một bài chỉ được giữ khi trả về đủ `referenceCount` của nó.
3. Các bài còn lại được đọc trang theo trang qua `/paper/{id}/references`.

Mỗi bài được gán một trạng thái: `success`, `empty` (có trong Semantic Scholar nhưng không liệt kê reference nào), `unavailable` (có reference nhưng không chia sẻ), `not_found` (chưa được lập chỉ mục), `api_error` (API lỗi liên tục, không ghi file, lần chạy sau thử lại). Nhờ vậy **reference rỗng thật** được phân biệt với **lỗi API**.

Với mỗi reference: arXiv ID được đọc từ `externalIds.ArXiv` (giữ nguyên dạng cũ trước 2007 như `hep-th/9901001`), cùng tiêu đề, tác giả, Semantic Scholar ID. `ref_submission_date` là ngày nộp `v1` trên arXiv, tra từ arXiv API hoặc từ bản metadata arXiv trên Kaggle. `publicationDate` của Semantic Scholar **không** được dùng thay ngày nộp arXiv. Mỗi reference còn có `metadata_status`: `complete` (có Semantic Scholar ID, tiêu đề và tác giả), `incomplete` hoặc `unresolved` (Semantic Scholar không liên kết được trích dẫn với một bài cụ thể).

Kết quả: `references_all.json` giữ **toàn bộ** reference, còn `references.json` chỉ giữ các reference có arXiv ID (tập con sẽ thành bảng `references` ở Milestone 3).

## 7. Giới hạn request và xử lý lỗi (`rate_limiter.py`)

| Máy chủ | Nhịp request | Xử lý lỗi |
| :--- | :--- | :--- |
| arXiv API | Tối đa 1 request mỗi 3 giây, 1 kết nối tại một thời điểm (theo API Terms of Use) | Retry có giới hạn; HTTP 429/503 tạm dừng **mọi** request tới arXiv, thời gian dừng tăng gấp đôi, tối đa 5 phút |
| `export.arxiv.org/src/` | Từng đợt 4 request cách nhau 0,25 s rồi nghỉ 1 s, theo trang *bulk data* của arXiv; tối đa 4 kết nối | Như trên; phiên bản lỗi được thử lại ở lượt sau |
| Semantic Scholar | 1 request/giây khi có API key, 1 request mỗi 3,1 giây khi không có | HTTP 429 chờ 2 giây (hoặc theo `Retry-After`), tối đa 12 lần; lỗi 5xx và lỗi mạng chờ 5, 10, 20, 40, 80 giây |

API key của Semantic Scholar được nạp từ biến môi trường `S2_API_KEY` (Colab Secrets) và không bao giờ được commit vào repository.

## 8. Kiểm tra dữ liệu và đo hiệu năng

- `validate.py` kiểm tra manifest, metadata, từng phiên bản (thư mục tồn tại, danh sách `kept_files` khớp với file trên đĩa, không còn file ngoài `.tex`/`.bib`, không có file rỗng) và hai file reference, xếp mỗi kết quả vào `PASS`, `WARN`, `PENDING` hoặc `FAIL` ([Chương 6](./06_errors_limitations.ipynb)).
- `corpus_statistics.py` tính các chỉ số từ trạng thái cuối cùng trên đĩa (`versions.json`, `references_all.json`), không cộng thẳng các dòng log vì log có cả các lần retry ([Chương 4](./04_crawling_results.ipynb)).
- `benchmark.py` chạy crawler như một tiến trình con, lấy mẫu RAM của cả cây tiến trình mỗi 2 giây bằng `psutil` và dung lượng thư mục dữ liệu mỗi 300 giây ([Chương 5](./05_performance.ipynb)).